In [ ]:
!pip3 install matplotlib

In [ ]:
!pip3 install pandas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.transforms import v2
from torch.functional import F
import torchvision.transforms.functional as TF

In [ ]:
from urllib.request import urlretrieve; urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_c0z0c_dev.py", "helper_c0z0c_dev.py")
import helper_c0z0c_dev as helper

In [ ]:
helper.setup()

# 데이터 불러오고 전처리

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# MPS 혹은 cuda가 사용 가능한지 확인
if torch.cuda.is_available():
    device = torch.device("cuda")

elif torch.backends.mps.is_available():
    device = torch.device("mps")

else:
    device = "cpu"
print(device)

In [ ]:
from glob import glob

In [ ]:
# 로컬에서 쓸 때

train_dir = glob('/Users/jayjun/Desktop/mission5_image/train/*.png')
train_cleaned_dir = glob('/Users/jayjun/Desktop/mission5_image/train_cleaned/*.png')
test_dir = glob('/Users/jayjun/Desktop/mission5_image/test/*.png')

# #1. 별 다른 데이터 증강없이

## 이미지 불러오기 및 전처리

In [ ]:


transform = v2.Compose(
    [
        v2.ToTensor(),
        v2.ToDtype(dtype=torch.float32, scale=True),

    ]
)

## 커스텀 이미지 데이터셋

In [ ]:
target_height = 420

class ImageDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path)
        current_weight, current_height = image.size
        if current_height == target_height:
            pass
        else:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(image)

        return transform(image)


In [ ]:
class PairedImageDataset(Dataset):
    def __init__(self, damaged_paths, clean_paths):
        self.damaged_paths = damaged_paths
        self.clean_paths = clean_paths

        # 경로 길이가 같은지 확인
        assert len(damaged_paths) == len(clean_paths)

        # 파일명으로 정렬해서 매칭 확실히 하기
        self.damaged_paths.sort()
        self.clean_paths.sort()

    def __len__(self):
        return len(self.damaged_paths)

    def __getitem__(self, idx):
        # 손상된 이미지 로드
        damaged_image = Image.open(self.damaged_paths[idx])
        current_width, current_height = damaged_image.size

        # 높이가 258인 경우 패딩 추가
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            damaged_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(damaged_image)

        # 타겟 이미지 로드
        clean_image = Image.open(self.clean_paths[idx])
        current_width, current_height = clean_image.size

        # 똑같이 패딩 적용
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            clean_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(clean_image)

        # transform 적용
        damaged_tensor, clean_tensor = transform(damaged_image, clean_image)
        # clean_tensor = transform(clean_image)

        return damaged_tensor, clean_tensor

# 데이터셋 생성
paired_train_dataset = PairedImageDataset(train_dir, train_cleaned_dir)
test_dataset = ImageDataset(test_dir)  # 테스트는 기존 커스텀 데이터셋 유지

print(f"학습 데이터셋 크기: {len(paired_train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

### 데이터셋 생성 및 이미지 데이터 시각화

In [ ]:
# 이미지 확인

train_dataset = ImageDataset(train_dir)
train_cleaned_dataset = ImageDataset(train_cleaned_dir)
test_dataset = ImageDataset(test_dir)


def show_images(index_for_train, index_for_test):
    plt.figure(figsize=(15, 5))
    for i in range(0,3):
        plt.subplot(1, 3, i+1)
        if i == 0:
            plt.imshow(train_dataset[index_for_train].squeeze(), cmap='gray')
        elif i == 1:
            plt.imshow(train_cleaned_dataset[index_for_train].squeeze(), cmap='gray')
        else:
            plt.imshow(test_dataset[index_for_test].squeeze(), cmap='gray')
        plt.axis('off')

    plt.show()

show_images(67, 53)


print(f'train_data 개수: {len(train_dataset)}개')
print(f'train_cleaned_data 개수 : {len(train_cleaned_dataset)}개')
print(f'test_data 개수 : {len(test_dataset)}개')

### 데이터 로더

In [ ]:
paired_train_loader = DataLoader(paired_train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

## 모델 구현

In [ ]:
class CNNAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(

            # 420×540 → 210×270
            nn.Conv2d(1, 32, 5, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 210×270 → 105×135
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 105×135 → 53×68
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            # 53x68 → 105x135
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 105×135 → 210×270
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 210×270 → 420×540
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        original_size = x.shape[2:]

        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        # 최종 크기 조정
        decoded = F.interpolate(decoded, size=original_size,
                              mode='bilinear', align_corners=False)

        return decoded, encoded

In [ ]:
# 모델 생성
CNNAEmodel = CNNAE().to(device)

### 학습 루프

In [ ]:
# 학습 루프
lr = 1e-3
epochs = 20
loss_fn = nn.MSELoss()
def train_loop(model, epochs, loss_fn=loss_fn, lr=lr):
    opt = optim.Adam(model.parameters(), lr = lr)
    step = 0
    for epoch in range(epochs):
        # 훈련 모드
        model.train()
        for batch_idx, (damaged, clean) in enumerate(paired_train_loader):
            damaged, clean = damaged.float().to(device), clean.float().to(device)
            decoded, _ = model(damaged)
            decoded = decoded.float()

            loss = loss_fn(decoded, clean)
            opt.zero_grad()
            loss.backward()
            opt.step()

            step+=1

            if step % 9 == 0: # 훈련 데이터가 총 144개, 배치 사이즈가 16이므로 각 에포크마다
                            # 9번의 가중치 업데이트가 진행되므로
                print(f'Epoch : {epoch+1}/{epochs}, Loss : {loss.item():.4f}')
    print("\n학습 종료")

In [ ]:
train_loop(CNNAEmodel, epochs)

### 평가 루프

In [ ]:
# 평가 루프

def evaluate_model(model, test_loader, device):
    model.eval()  # 평가 모드로 설정

    total_loss = 0
    num_batches = 0

    # 결과 저장용
    last_inputs = []
    last_outputs = []

    with torch.no_grad():
        for batch_idx, test_images in enumerate(test_loader):
            test_images = test_images.to(device)

            # 학습된 모델로 추론
            decoded_images, _ = model(test_images)

            # 자기 자신과 비교해서 복원 성능 측정

            num_batches += 1

            # 배치 중 랜덤한 배치의 결과 저장(시각화용)
            if batch_idx == 7:
                last_inputs.append(test_images.cpu())
                last_outputs.append(decoded_images.cpu())

    print("테스트 이미지 복원 완료")
    return last_inputs, last_outputs

In [ ]:
# 평가 실행

test_inputs, test_outputs = evaluate_model(CNNAEmodel, test_loader, device)

### 결과 시각화

In [ ]:
# 결과 시각화
def show_test_results(inputs, outputs, num_samples=4):
    input_batch = inputs[0]
    output_batch = outputs[0]
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 10))

    for i in range(num_samples):
        # 원본 (테스트) 이미지
        axes[0, i].imshow(input_batch[i].squeeze(), cmap='gray')
        axes[0, i].set_title(f'Original {i+1}')
        axes[0, i].axis('off')

        # 복원된 이미지
        axes[1, i].imshow(output_batch[i].squeeze(), cmap='gray')
        axes[1, i].set_title(f'Reconstructed {i+1}')
        axes[1, i].axis('off')

    plt.suptitle('Reconstructions of test images', fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
# 결과 확인
show_test_results(test_inputs, test_outputs)

## 데이터 증강 없이 진행한 결과

첫 시도로 Convolution Layer를 사용하는 오토인코더 모델을 사용하였다.

리니어 레이어로 이루어진 모델에는 입력 개수가 너무 많아져 연산량이 많아짐에 따라 학습의 속도가 너무 느릴 것으로 예상했기 때문이다.

그리고 풀링 레이어를 제외하였는데 풀링을 포함한 모델의 결과를 보니 복원이 제대로 되지 않았고 이 과제의 목표는 노이즈를 제거하는 것이기 때문에 이미지의 세세한 특징을 최대한 학습 시켜야한다는 생각에 풀링을 진행하지 않았고 커널 사이즈도 모든 컨볼루션 레이어에서 3으로 바꾸고 스트라이드도 1로 설정하였다. 여러번 모델 학습을 진행해본 결과, 오차가 꾸준히 내려가는 것으로 보아 학습이 문제없이 잘 진행된 것으로 보이고, 테스트 이미지를 복원해서 시각화해본 결과도 준수했다.

# #2. 여러 데이터 증강 적용



## 이미지 증강 (로테이션: 10, 밝기: 0.5, 대비: 0.5)

In [ ]:
augmented_transform = v2.Compose([
    v2.RandomRotation(degrees=5),
    v2.ColorJitter(brightness=0.5 , contrast=0.5),
    v2.ToTensor(),
    v2.ToDtype(dtype=torch.float32, scale=True),
])

In [ ]:
class PairedImageDataset(Dataset):
    def __init__(self, damaged_paths, clean_paths):
        self.damaged_paths = damaged_paths
        self.clean_paths = clean_paths

        # 경로 길이가 같은지 확인
        assert len(damaged_paths) == len(clean_paths)

        # 파일명으로 정렬해서 매칭 확실히 하기
        self.damaged_paths.sort()
        self.clean_paths.sort()

    def __len__(self):
        return len(self.damaged_paths)

    def __getitem__(self, idx):
        # 손상된 이미지 로드
        damaged_image = Image.open(self.damaged_paths[idx])
        current_width, current_height = damaged_image.size

        # 높이가 258인 경우 패딩 추가
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            damaged_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(damaged_image)

        # 타겟 이미지 로드 (같은 전처리)
        clean_image = Image.open(self.clean_paths[idx])
        current_width, current_height = clean_image.size

        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            clean_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(clean_image)

        # transform 적용
        # damaged_tensor = augmented_transform(damaged_image)
        # clean_tensor = augmented_transform(clean_image)

        damaged_tensor, clean_tensor = augmented_transform(damaged_image, clean_image)

        # damaged_tensor = v2.ToTensor()(damaged_tensor)
        # clean_tensor = v2.ToTensor()(clean_tensor)

        # damaged_tensor = v2.ToDtype(dtype=torch.float32, scale=True)(damaged_tensor)
        # clean_tensor = v2.ToDtype(dtype=torch.float32, scale=True)(clean_tensor)
        return damaged_tensor, clean_tensor

# 데이터셋 생성
paired_train_dataset = PairedImageDataset(train_dir, train_cleaned_dir)
test_dataset = ImageDataset(test_dir)  # 테스트는 기존 커스텀 데이터셋 유지

In [ ]:
paired_train_loader = DataLoader(paired_train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
model = CNNAE().to(device)

In [ ]:
# 학습 루프
lr = 1e-3
epochs = 30
loss_fn = nn.MSELoss()
train_loop(model, epochs,loss_fn)

In [ ]:
# 모델 평가 실행
print("최종 테스트 결과")
test_inputs, test_outputs, test_loss = evaluate_model(model, test_loader, device)

### 첫번째 이미지 증강 결과

In [ ]:
# 결과 확인
show_test_results(test_inputs, test_outputs)

## 이미지 증강2 (로테이션: 5, 밝기: 0.5, 대비: 0.5)

In [ ]:
augmented_transform2 = v2.Compose(
            [

            v2.RandomRotation(degrees=5),
            v2.ColorJitter(brightness=0.5 , contrast=0.5),] # 밝기, 대비 변경
)

In [ ]:
class PairedImageDataset2(Dataset):
    def __init__(self, damaged_paths, clean_paths):
        self.damaged_paths = damaged_paths
        self.clean_paths = clean_paths

        # 경로 길이가 같은지 확인
        assert len(damaged_paths) == len(clean_paths)

        # 파일명으로 정렬해서 매칭 확실히 하기
        self.damaged_paths.sort()
        self.clean_paths.sort()

    def __len__(self):
        return len(self.damaged_paths)

    def __getitem__(self, idx):
        # 손상된 이미지 로드
        damaged_image = Image.open(self.damaged_paths[idx])
        current_width, current_height = damaged_image.size

        # 높이가 258인 경우 패딩 추가
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            damaged_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(damaged_image)

        # 타겟 이미지 로드 (같은 전처리)
        clean_image = Image.open(self.clean_paths[idx])
        current_width, current_height = clean_image.size

        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            clean_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(clean_image)

        # transform 적용
        damaged_tensor, clean_tensor = augmented_transform2(damaged_image, clean_image)

        damaged_tensor = v2.ToTensor()(damaged_tensor)
        clean_tensor = v2.ToTensor()(clean_tensor)

        damaged_tensor = v2.ToDtype(dtype=torch.float32, scale=True)(damaged_tensor)
        clean_tensor = v2.ToDtype(dtype=torch.float32, scale=True)(clean_tensor)

        return damaged_tensor, clean_tensor

# 데이터셋 생성
paired_train_dataset = PairedImageDataset2(train_dir, train_cleaned_dir)

test_dataset = ImageDataset(test_dir)  # 테스트는 기존 커스텀 데이터셋 유지

In [ ]:
paired_train_loader = DataLoader(paired_train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
model2 = CNNAE().to(device)

In [ ]:
# 학습 루프
lr = 1e-3
epochs = 20
loss_fn = nn.MSELoss()

train_loop(model2, epochs)

In [ ]:

# 모델 평가 실행
print("최종 테스트 결과")
test_inputs, test_outputs, test_loss = evaluate_model(model2, test_loader, device)

### 두번째 이미지 증강 결과

In [ ]:
# 결과 시각화
def show_test_results(inputs, outputs, num_samples=4):
    input_batch = inputs[0]
    output_batch = outputs[0]
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 10))

    for i in range(num_samples):
        # 테스트 이미지
        axes[0, i].imshow(input_batch[i].squeeze(), cmap='gray')
        axes[0, i].set_title(f'테스트 입력 {i+1}')
        axes[0, i].axis('off')

        # 복원 이미지
        axes[1, i].imshow(output_batch[i].squeeze(), cmap='gray')
        axes[1, i].set_title(f'복원된 출력 {i+1}')
        axes[1, i].axis('off')

    plt.suptitle('테스트 이미지 복원 결과', fontsize=16)
    plt.tight_layout()
    plt.show()

# 결과 확인
show_test_results(test_inputs, test_outputs)

## 이미지 증강3  (밝기: 0.5, 대비: 0.5)

In [ ]:
augmented_transform3 = v2.Compose(
    [

        v2.ColorJitter(brightness=0.5, contrast = 0.5),
        v2.ToTensor(),
        v2.ToDtype(dtype=torch.float32, scale=True),
    ]
)

In [ ]:
class PairedImageDataset3(Dataset):
    def __init__(self, damaged_paths, clean_paths):
        self.damaged_paths = damaged_paths
        self.clean_paths = clean_paths

        # 경로 길이가 같은지 확인
        assert len(damaged_paths) == len(clean_paths)

        # 파일명으로 정렬해서 매칭 확실히 하기
        self.damaged_paths.sort()
        self.clean_paths.sort()

    def __len__(self):
        return len(self.damaged_paths)

    def __getitem__(self, idx):
        # 손상된 이미지 로드
        damaged_image = Image.open(self.damaged_paths[idx])
        current_width, current_height = damaged_image.size

        # 높이가 258인 경우 패딩 추가
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            damaged_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(damaged_image)

        # 타겟 이미지 로드 (같은 전처리)
        clean_image = Image.open(self.clean_paths[idx])
        current_width, current_height = clean_image.size

        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            clean_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(clean_image)


        # transform 적용
        damaged_tensor, clean_tensor = augmented_transform3(damaged_image, clean_image)

        return damaged_tensor, clean_tensor

# 데이터셋 생성
paired_train_dataset = PairedImageDataset3(train_dir, train_cleaned_dir)

test_dataset = ImageDataset(test_dir)  # 테스트는 기존 커스텀 데이터셋 유지

In [ ]:
# 데이터 로더

paired_train_loader = DataLoader(paired_train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
model3 = CNNAE().to(device)

In [ ]:
# 학습 루프
epochs = 30
train_loop(model3, epochs)

In [ ]:
# 모델 평가 실행
print("최종 테스트 결과")
test_inputs, test_outputs, test_loss = evaluate_model(model3, test_loader, device)

### 세번째 이미지 증강 결과

In [ ]:
# 결과 확인
show_test_results(test_inputs, test_outputs)

## 이미지 증강4 (밝기: 0.3, 대비: 0.3)

In [ ]:
augmented_transform4 = v2.Compose(
    [
        v2.ColorJitter(brightness=0.3 , contrast=0.3),
        v2.ToTensor(),
        v2.ToDtype(dtype=torch.float32, scale=True),
    ]
)

In [ ]:
class PairedImageDataset4(Dataset):
    def __init__(self, damaged_paths, clean_paths):
        self.damaged_paths = damaged_paths
        self.clean_paths = clean_paths

        # 경로 길이가 같은지 확인
        assert len(damaged_paths) == len(clean_paths)

        # 파일명으로 정렬해서 매칭 확실히 하기
        self.damaged_paths.sort()
        self.clean_paths.sort()

    def __len__(self):
        return len(self.damaged_paths)

    def __getitem__(self, idx):
        # 손상된 이미지 로드
        damaged_image = Image.open(self.damaged_paths[idx])
        current_width, current_height = damaged_image.size

        # 높이가 258인 경우 패딩 추가
        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            damaged_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(damaged_image)

        # 타겟 이미지 로드 (같은 전처리)
        clean_image = Image.open(self.clean_paths[idx])
        current_width, current_height = clean_image.size

        if current_height != target_height:
            needed_padding = target_height - current_height
            upper_padding = needed_padding // 2
            lower_padding = needed_padding - upper_padding
            clean_image = v2.Pad((0, upper_padding, 0, lower_padding), fill=255)(clean_image)


        # transform 적용
        damaged_tensor, clean_tensor = augmented_transform4(damaged_image, clean_image)

        return damaged_tensor, clean_tensor

# 데이터셋 생성
paired_train_dataset = PairedImageDataset4(train_dir, train_cleaned_dir)

test_dataset = ImageDataset(test_dir)  # 테스트는 기존 커스텀 데이터셋 유지

In [ ]:
paired_train_loader = DataLoader(paired_train_dataset, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
model4 = CNNAE().to(device)

In [ ]:
epochs = 50
train_loop(model4, epochs)

In [ ]:
print("최종 테스트 결과")
test_inputs, test_outputs, test_loss = evaluate_model(model4, test_loader, device)

### 네번째 이미지 증강 결과

In [ ]:
show_test_results(test_inputs, test_outputs)

## 데이터 증강 적용 시도 결과

데이터를 증강시킴에 따라 모델이 더 강건해질 수 있나 보고 싶었다. 첫번째와 두번째 이미지 증강에는 RandomRotation과 밝기와 대비에 변화를 주기 위해 Colorjitter를 적용했다. 텍스트가 적혀있는 데이터이기 때문에 위 아래 혹은 양 옆으로 뒤집는 방법은 좋지 않다고 생각했다. 이렇게 사진을 반전을 시키면 왼쪽에서 오른쪽으로 읽히는 텍스트의 특징에 대한 학습이 잘 되지 않을 것이라고 생각해서, 약간의 Rotation을 주고 밝기와 대비에 약한 변화만 주도록 했다.

대체적으로 이미지에 아무런 변화를 주지 않았을 때 보다 Epoch 수를 올려서 학습을 진행해야 하는 것으로 보인다. 별 다른 데이터 증강 없이 시도했을 때는 Epoch을 20 이하로 정하고 학습 시킨 모델로 테스트 이미지를 복구시켜도 복구가 잘 되었다. 그러나 로테이션과 밝기, 대비에 변화를 준 이미지들로 모델을 훈련시킬 때에는 Epoch을 100, 5배 더 많은 학습을 진행해도 아무런 증강을 적용하지 않은 데이터로 학습시킨 모델의 결과 만큼 좋은 결과를 얻지 못했다.

# #3. 다양한 하이퍼 파라미터 적용

학습률, 손실 함수, 에포크 수, 모델 구성 등의 다양한 하이퍼 파라미터를 변경시켜보며 나오는 결과 확인

In [ ]:
# 공통적으로 사용할 데이터셋

common_train_paired = PairedImageDataset(train_dir, train_cleaned_dir)
common_test = ImageDataset(test_dir)

In [ ]:
# 공통적으로 사용할 데이터로더
paired_train_loader = DataLoader(common_train_paired, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(common_test, batch_size=8, shuffle=False)

## #1) 모델에 변화 주기

### #1-1) 배치 정규화의 유무가 결과에 미치는 영향 확인

**손실함수: MSE, 학습률: 0.0001, 에포크 수: 30 고정**

### Without 배치 정규화 시작

In [ ]:
# 배치 정규화를 하지 않는 모델 구현
class CNN_AE_without_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(

            # 420×540 → 210×270
            nn.Conv2d(1, 32, 5, stride=2, padding=1),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 210×270 → 105×135
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 105×135 → 53×68
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            # 53x68 → 105x135
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1),
            nn.ReLU(),

            # 105×135 → 210×270
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            # 210×270 → 420×540
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        original_size = x.shape[2:]

        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        # 최종 크기 조정
        decoded = F.interpolate(decoded, size=original_size,
                              mode='bilinear', align_corners=False)

        return decoded, encoded

In [ ]:
noBN_model = CNN_AE_without_BN().to(device)

In [ ]:
lr = 1e-3
epochs = 30
loss_fn = nn.MSELoss()
train_loop(noBN_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(noBN_model, test_loader, device)

### Without 배치 정규화 결과

In [ ]:
show_test_results(test_inputs, test_outputs, num_samples=4)

### With 배치 정규화 시작

In [ ]:
CNNAE_withBN = CNNAE().to(device)

In [ ]:
lr = 1e-3
epochs = 30
loss_fn = nn.MSELoss()
train_loop(CNNAE_withBN, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs= evaluate_model(CNNAE_withBN, test_loader, device)

### With 배치 정규화 결과

In [ ]:
show_test_results(test_inputs, test_outputs, num_samples=4)

### 배치 정규화의 영향
수치적으로 보았을 때, 배치 정규화가 진행될 때의 loss는 진행되지 않을 때에 비해 더 낮았다. 두 모델 모두 접힘, 구겨짐과 같은 손상은 잘 복구했다고 볼 수 있을 것 같지만, 복원된 이미지를 보면, 배치 정규화가 진행되지 않은 모델로 복원한 이미지는 각기 다른 글씨체를 제대로 학습을 하지 못한 모습이 종종 보인다. 반대로 배치 정규화 모델은 테스트 입력으로 들어간 이미지의 각기 다른 글씨체도 학습한 모습이다. 배치 정규화는 세세한 디테일을 학습하는데 도움을 줄 수 있는 것 같다.

### #1-2) Convolution Layer에 변화 줘보기

In [ ]:
class CNNAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(

            # 420×540 → 210×270
            nn.Conv2d(1, 32, 5, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 210×270 → 105×135
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # nn.MaxPool2d(2,2),

            # 105×135 → 53×68
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()

            nn.Conv2d(128, 256, 3, stride = 1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLu()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 53x68 → 105x135
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # 105×135 → 210×270
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 210×270 → 420×540
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        original_size = x.shape[2:]

        encoded = self.encoder(x)
        decoded = self.decoder(encoded)

        # 최종 크기 조정
        decoded = F.interpolate(decoded, size=original_size,
                              mode='bilinear', align_corners=False)

        return decoded, encoded

## #2) 학습/평가 과정에서 변화 주기

### #2-1) 에포크 수가 미치는 결과에 미치는 영향 확인

Epoch을 30, 50, 100으로 두고 똑같은 데이터셋, 학습률, 손실 함수를 사용하여 에포크 수의 영향 확인

In [ ]:
# 공통적으로 사용할 모델
common_cnn_model = CNNAE().to(device)

In [ ]:
common_train_paired = PairedImageDataset(train_dir, train_cleaned_dir)
common_test = ImageDataset(test_dir)

In [ ]:
# 공통적으로 사용할 데이터로더
paired_train_loader = DataLoader(common_train_paired, batch_size=16, shuffle=True, drop_last=True)
test_loader = DataLoader(common_test, batch_size=8, shuffle=False)

### 에포크 30

In [ ]:
# 에포크 30으로 학습 시작

lr = 0.0001
loss_fn = nn.MSELoss()
epochs = 30
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
# 에포크가 30인 학습을 위한 평가 함수
test_inputs, test_outputs= evaluate_model(common_cnn_model, test_loader, device)

In [ ]:
# 에포크가 30인 학습을 위한 결과 시각화 함수
show_test_results(test_inputs, test_outputs, num_samples=5)

### 에포크 50

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 에포크 50으로 학습 시작

lr = 0.0001
loss_fn = nn.MSELoss()
epochs = 50
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
# 에포크가 50인 학습을 위한 평가 함수
test_inputs, test_outputs = evaluate_model(common_cnn_model, test_loader, device)

In [ ]:
# 에포크가 50인 학습을 위한 결과 시각화 함수
show_test_results(test_inputs, test_outputs, num_samples=5)

### 에포크 100

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 에포크 100으로 학습 시작

lr = 0.0001
loss_fn = nn.MSELoss()
epochs = 100
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
# 에포크가 100인 학습을 위한 평가 함수
test_inputs, test_outputs  = evaluate_model(common_cnn_model, test_loader, device)

In [ ]:
# 에포크가 100인 학습을 위한 결과 시각화 함수
show_test_results(test_inputs, test_outputs, num_samples=4)

### 에포크 수의 영향

에포크 수를 점점 올릴 수록 최종 훈련 오차가 낮아지는 것을 볼 수 있었다. 손상된 이미지의 디테일을 늘어난 에포크 수 만큼 더 학습 할 수 있는 것이기 때문에 에포크 수가 클수록 훈련 오차가 낮은 것은 어찌보면 당연한 것 같다. 워낙 훈련 데이터 자체가 적고, 30이나 50 에포크의 학습과정에서는 충분히 손상의 디테일을 학습하지 못해 loss가 상대적으로 높고, 가중치 초기값을 안 좋게 설정하는 경우에는 평소의 첫 epoch에 계산된 loss의 3배 이상까지도 보이는 경우가 있었다. 나는 에포크 30, 50, 100의 경우만을 실험해봤지만, 데이터의 양이 매우 적기 때문에, epoch을 더 올려서 너무 많이 학습을 진행하게 되면 과적합이 될 수 있을 것 같다.

### #2-2) 학습률 변화 주기

손실 함수: MSE, 옵티마이저: Adam, 에포크 수: 20 고정 후, 학습률 변화에 따른 차이 보기

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
lr_exp_train_paired = PairedImageDataset(train_dir, train_cleaned_dir)
lr_exp_test = ImageDataset(test_dir)

In [ ]:
paired_train_loader = DataLoader(lr_exp_train_paired, batch_size=16, shuffle=False, drop_last=True)
lr_test_loader = DataLoader(lr_exp_test, batch_size=8, shuffle=False)

### 학습률 1

In [ ]:
lr = 1
epochs = 20
loss_fn = nn.MSELoss()
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(common_cnn_model, lr_test_loader, device)

In [ ]:
show_test_results(test_inputs, test_outputs)

학습률을 1로 두니까 아예 학습이 진행되지 않았다.

### 학습률 0.1

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 학습률 0.1 학습

lr = 0.1
epochs = 20
loss_fn = nn.MSELoss()
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(common_cnn_model, lr_test_loader, device)

In [ ]:
show_test_results(test_inputs, test_outputs)

### 학습률 0.01

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 학습률 0.01

lr = 0.01
epochs = 20
loss_fn = nn.MSELoss()
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(common_cnn_model, lr_test_loader, device)

In [ ]:
show_test_results(test_inputs, test_outputs)

### 학습률 0.001

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 학습률 0.001

lr = 0.001
epochs = 20
loss_fn = nn.MSELoss()
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(common_cnn_model, lr_test_loader, device)

In [ ]:
show_test_results(test_inputs, test_outputs)

### 학습률 0.0001

In [ ]:
# 모델 초기화
common_cnn_model = CNNAE().to(device)

In [ ]:
# 학습률 0.0001

lr = 0.0001
# epochs = 20
# epochs을 두 배로 늘려 보면 결국 전역 최고점을 찾을 수 있을까?
epochs = 40
loss_fn = nn.MSELoss()
train_loop(common_cnn_model, epochs, loss_fn, lr)

In [ ]:
test_inputs, test_outputs = evaluate_model(common_cnn_model, lr_test_loader, device)

In [ ]:
show_test_results(test_inputs, test_outputs)

### 학습률 변화가 결과에 주는 영향

학습률을 1부터 0.0001까지 0.1씩 곱해가며 바꿔봤다. 그 결과, 학습률 1에서는 전혀 학습이 진행되지 않는 모습이었다. 복원된 이미지를 시각화해서 봐도 알 수 있었고, 훈련과정에서의 loss를 보게 되면 첫 epoch의 loss에서 (-0.015 ~ +0.015) 정도의 범위 내에서 진동하는 것을 볼 수 있었고 학습이 점점 진행됨에 따라 다른 학습률의 경우에는 중간에 loss값이 갑자기 솟을 때가 있었지만 그 이후에 점점 낮아지는 경우가 대부분이었지만 학습률 1의 경우에는 그러지 않았다. 계속 비슷한 범위 내에서 진동만 할 뿐이었다. 그리고 복원된 이미지를 보니 검은색 사각형만 출력되었다.

train_loop 함수에서 출력되는 첫번째 loss는 한 에포크를 학습하고 난 후의 오차인데 학습률 0.1에서는 그 첫 오차가 꾸준히 0.0600 ~ 0.0700 범위 안의 값으로 측정되었다. (+ loss가 모델의 성능을 대표하기 보단, loss는 훈련의 과정이 잘 진행되고 있는지를 판단하는 지표로 사용하는 것이 더 바람직하다.) 학습이 다 진행되고 최종적으로 나온 훈련 loss는 0.0300 ~ 0.0400의 사잇값으로 꾸준하게 나왔다. 지금까지 미션을 진행하면서 한 가지 모델만으로 진행했는데 훈련과정에서 나온 가장 낮은 loss는 0.0295 정도의 값이었다. 그래서 epoch 20으로 두고 학습률 0.1로 진행한 학습이 훈련오차가 최종적으로 0.0339가 나온 것은 훈련이 잘 된 증거라고 생각했다. 하지만 테스트 이미지를 모델에 입력시켜서 복원해보니 복원이 잘 되지 않았다. 이것이 훈련 데이터에만 지나치게 학습이 된 과적합으로 보인다. 이걸로 훈련 오차가 낮다는 것이 좋은 일반화 성능을 뜻하는 것이 아님을 배웠다.

5가지의 학습률 실험에서 가장 결과가 좋게 나온 것은 학습률 0.01 과 0.001이었다. 두 학습률 모두 loss도 첫 loss에서 점차 계속해서 떨어지는 모습을 볼 수 있고, 현재 모델이 과적합 되었는지 확인 할 방법은 테스트 이미지를 직접 넣어 복원 성능을 확인해 보는 것 인데, 0.01로 학습한 모델, 0.001로 학습을 진행한 모델 모두 복원 성능이 좋았다. 어떤 모델이 더 잘 복원했냐고 물으면, 입력값으로 들어간 테스트 이미지가 무엇이냐에 따라 의견이 갈릴 것 같다. 내 의견은 0.001 모델이 구겨짐을 더 잘 복구한 것 같고, 0.01 모델은 글자가 선명하지 않은 것이 있는데, 0.001 모델은 대부분의 텍스트가 선명하게 보인다.

학습률이 0.0001인 경우에는 학습이 잘 진행되지 않았다. 덜 진행되었다고 보는 것이 맞는 것 같다. loss가 하락 하는 것을 보면 점차 하락하지만 그 폭이 크지 않아서 step이 너무 작게 설정되었음을 제대로 확인 할 수 있다. 옳은 방향으로 가중치 계산을 하고 있지만 step이 너무 작아서 전역 최소점을 찾는데 시간이 더 걸리는 것으로 보인다. Epoch을 좀 더 늘리면 모델의 일반화 성능이 어떻게 되는지 확인해보겠다.

학습률 0.0001의 epochs를 2배인 40으로 늘린 결과 -> epochs가 20으로 설정되었을 때보다 훨씬 좋은 복원 성능을 보여주었다. loss도 한번도 오르지 않고 조금씩 천천히 계속 떨어지는 모습을 보여주었다. 느리지만 학습은 제대로 하게 되는 것으로 보인다. 더 늘려서 실험 해보고 싶은데 코랩 컴퓨팅 유닛이 부족해서 다음으로 넘어가겠다.

## #4. 검증 데이터로 모델 성능 검증 과정 추가

In [ ]:
!pip3 install scikit-learn

### 훈련 데이터를 분할하여 검증 데이터로 만들기

In [ ]:
from sklearn.model_selection import train_test_split

# 훈련 데이터를 train/validation으로 분할
train_damaged, val_damaged, train_clean, val_clean = train_test_split(
    train_dir, train_cleaned_dir, test_size=0.3, random_state=42
)

paired_train_dataset = PairedImageDataset(train_damaged, train_clean)
paired_val_dataset = PairedImageDataset(val_damaged, val_clean)

In [ ]:
print(len(paired_train_dataset))

In [ ]:
print(len(paired_val_dataset))

In [ ]:
# 데이터 로더로 데이터셋 배치 사이즈만큼 나누기
paired_train_loader = DataLoader(paired_train_dataset, batch_size=10, shuffle=True)
paired_val_loader = DataLoader(paired_val_dataset, batch_size=4, shuffle=False)

In [ ]:
cnn_model = CNNAE().to(device)

### 훈련 시작

훈련 데이터 총 개수: 100개

배치 사이즈: 10개

배치 총 개수: 10개

In [ ]:
lr = 1e-3
epochs = 20
loss_fn = nn.MSELoss()
def train_loop(model, epochs, loss_fn=loss_fn, lr=lr):
    opt = optim.Adam(model.parameters(), lr = lr)
    step = 0
    for epoch in range(epochs):
        # 훈련 모드
        model.train()
        for batch_idx, (damaged, clean) in enumerate(paired_train_loader):
            damaged, clean = damaged.float().to(device), clean.float().to(device)
            decoded, _ = model(damaged)
            decoded = decoded.float()

            loss = loss_fn(decoded, clean)
            opt.zero_grad()
            loss.backward()
            opt.step()

            step+=1

            if step % 10 == 0: # 훈련 데이터가 총 100개, 배치 사이즈가 10이므로 각 에포크마다
                            # 10번의 가중치 업데이트가 진행되므로
                print(f'Epoch : {epoch+1}/{epochs}, Loss : {loss.item():.4f}')
    print("학습 종료")

In [ ]:
train_loop(cnn_model, epochs, loss_fn)

### 검증 시작

검증 데이터 총 개수: 44개

배치 사이즈:  4개

배치 총 개수:  11개

In [ ]:
def validate_model(model, test_loader, device):
    # 평가 모드로 설정
    model.eval()

    total_loss = 0
    num_batches = 0

    # 결과 저장용
    last_inputs = []
    last_outputs = []

    with torch.no_grad():
        for batch_idx, (val_damaged, val_cleaned) in enumerate(paired_val_loader):
            val_damaged = val_damaged.to(device)
            val_cleaned = val_cleaned.to(device)
            # 학습된 모델로 추론
            decoded_images, _ = model(val_damaged)

            loss = loss_fn(decoded_images, val_cleaned)
            total_loss += loss

            num_batches += 1

            if batch_idx == 7:
                last_inputs.append(val_damaged.cpu())
                last_outputs.append(decoded_images.cpu())

            print(f'{batch_idx+1}번째 검증, Loss: {loss.item():.4f}')

    print('모델 검증 종료')
    print(f'검증 평균 Loss: {(total_loss/num_batches):.4f}')

    return last_inputs, last_outputs

In [ ]:
test_inputs, test_outputs = validate_model(cnn_model, paired_val_loader, device)

데이터가 너무 적지만, 훈련 데이터를 7:3으로 분할하여 44개의 데이터를 검증 데이터로 사용했다. 훈련 loss와 검증 loss가 0.0314로 큰 차이가 없다. 모델이 적절히 학습했고 과적합이 발생하지 않았음을 볼 수 있다.

### 검증 마친 모델로 테스트 이미지 복원

In [ ]:
# 테스트 이미지 불러오기

test_dataset2 = ImageDataset(test_dir)
test_loader = DataLoader(test_dataset2, batch_size=8, shuffle=False)

In [ ]:
# 만들어뒀던 테스트 함수 사용해서 테스트 이미지 복원

test_inputs, test_outputs = evaluate_model(cnn_model, test_loader, device)

In [ ]:
# 복원한 결과 보기

show_test_results(test_inputs, test_outputs)